# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score, precision_score, recall_score, roc_auc_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
import joblib
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
from itertools import product
from tqdm import tqdm
import itertools

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [3]:
df = pd.concat([pd.read_csv('../data/day-of-week-not-scaled.csv'), pd.read_csv('../data/dayofweek.csv')['dayofweek']], axis=1)
X = df.drop(columns='dayofweek')
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [40]:
svc = SVC(random_state=21, kernel='rbf', C=10, gamma='auto', class_weight=None, probability=True)

svc.fit(X_train, y_train)

y_pred = svc.predict(X_test)
y_prob = svc.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")

accuracy is 0.84911
precision is 0.85390
recall is 0.84911


In [5]:
dt_model = DecisionTreeClassifier(random_state=21, criterion='gini', max_depth=21, class_weight='balanced')

dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_test)
y_prob = dt_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")

accuracy is 0.88462
precision is 0.88765
recall is 0.88462


In [42]:
rf_model = RandomForestClassifier(random_state=21, n_estimators=100, criterion='entropy', max_depth=24, class_weight='balanced')

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")

accuracy is 0.88757
precision is 0.89092
recall is 0.88757


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [ ]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df['dayofweek'] = pd.read_csv("../data/dayofweek.csv")["dayofweek"]
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns="dayofweek"), df["dayofweek"], test_size=0.2, random_state=21, stratify=df["dayofweek"])
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

In [ ]:
svc_model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
tree_model = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
forest_model = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)
first = [1, 1, 1]

res_df = pd.DataFrame(columns=['accuracy', 'precision'])

for vote in ['soft', 'hard']:
    for first_w in range(1, 6):
        for second_w in range(1, 6):
            for third_w in range(1, 6):
                voting_model = VotingClassifier(estimators=(('SVC', svc_model), ('tree', tree_model), ('forest', forest_model)), voting=vote, weights=[first_w, second_w, third_w])
                voting_model.fit(X_train, y_train)
                prediction = voting_model.predict(X_val)
                acc_score = accuracy_score(y_val, prediction)
                pre_score = precision_score(y_val, prediction, average='weighted')
                res_df.loc[f'{first_w} {second_w} {third_w} {vote}'] = {'accuracy': acc_score, 'precision': pre_score}

display(res_df)


,accuracy,precision
1 1 1 soft,0.885185,0.888402
1 1 2 soft,0.900000,0.901802
1 1 3 soft,0.896296,0.897795
1 1 4 soft,0.900000,0.900226
1 1 5 soft,0.900000,0.900226
...,...,...
5 5 1 hard,0.903704,0.903584
5 5 2 hard,0.903704,0.903584
5 5 3 hard,0.903704,0.903584
5 5 4 hard,0.903704,0.903584


In [34]:
res_df = res_df.sort_values(by=["accuracy", "precision"], ascending=[False, False])
res_df

,accuracy,precision
4 1 4 soft,0.911111,0.912881
4 1 5 soft,0.911111,0.911443
5 1 1 soft,0.907407,0.911495
5 1 2 soft,0.907407,0.911495
4 1 3 soft,0.907407,0.910989
...,...,...
1 5 3 hard,0.866667,0.871697
2 4 1 hard,0.866667,0.871697
2 5 1 hard,0.866667,0.871697
2 5 2 hard,0.866667,0.871697


In [56]:
voting_model = VotingClassifier(estimators=(('SVC', svc_model), ('tree', tree_model), ('forest', forest_model)), voting='soft', weights=[4, 1, 4])
voting_model.fit(X_train, y_train)
prediction = voting_model.predict(X_test)
acc_score = accuracy_score(y_test, prediction)
pre_score = precision_score(y_test, prediction, average='weighted')
print(f'accuracy: {acc_score}, precision: {pre_score}')

accuracy: 0.908284023668639, precision: 0.9105141931507544


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [68]:
svc_model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
res_df = pd.DataFrame(columns=['accuracy', 'precision', 'recall'])
for i in range(1, 60):
    bagging_model = BaggingClassifier(base_estimator=SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True), random_state=21, n_estimators=i, n_jobs=-2)
    bagging_model.fit(X_train, y_train)
    prediction = bagging_model.predict(X_test)
    acc_score = accuracy_score(y_test, prediction)
    pre_score = precision_score(y_test, prediction, average='weighted')
    rec_score = recall_score(y_test, prediction, average="weighted")
    res_df.loc[i] = {'accuracy': acc_score, 'precision': pre_score, 'recall': rec_score}

In [70]:
res_df = res_df.sort_values(by=["accuracy", "precision"], ascending=[False, False])
res_df.head(1)

,accuracy,precision,recall
50,0.884615,0.889413,0.884615


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [ ]:
svc_model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
tree_model = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
forest_model = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)

In [67]:
estimators=[
            ('dt', tree_model),
            ('svc', svc_model),
            ('rf', forest_model)
        ]

n_splits_list = [2, 3, 4, 5, 6, 7]
passthrough_list = [True, False]

experiments = []

for n_splits in n_splits_list:
    for passthrough in passthrough_list:
        
        cv_gen = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        
        stacking_model = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(solver='liblinear', random_state=21),
            cv=cv_gen,
            passthrough=passthrough,
            n_jobs=-1
        )
        
        stacking_model.fit(X_train, y_train)
        val_preds = stacking_model.predict(X_val)
        
        acc = accuracy_score(y_val, val_preds)
        prec = precision_score(y_val, val_preds, average='weighted', zero_division=0)
        rec = recall_score(y_val, val_preds, average='weighted', zero_division=0)
        
        experiments.append({
            'n_splits': n_splits,
            'passthrough': passthrough,
            'val_acc': acc,
            'val_prec': prec,
            'val_rec': rec,
            'model': stacking_model
        })

best_experiment = sorted(
    experiments,
    key=lambda x: (x['val_acc'], x['val_prec']),
    reverse=True
)[0]

print("=== ЛУЧШИЕ ПАРАМЕТРЫ СТЕКИНГА (по Validation) ===")
print(f"n_splits:    {best_experiment['n_splits']}")
print(f"passthrough: {best_experiment['passthrough']}")
print(f"Val Accuracy:  {best_experiment['val_acc']:.4f}")
print(f"Val Precision: {best_experiment['val_prec']:.4f}\n")

best_stacking_model = best_experiment['model']
test_preds = best_stacking_model.predict(X_test)

print("=== ФИНАЛЬНЫЕ МЕТРИКИ НА TEST SET ===")
print(f"Test Accuracy:  {accuracy_score(y_test, test_preds):.4f}")
print(f"Test Precision: {precision_score(y_test, test_preds, average='weighted', zero_division=0):.5f}")
print(f"Test Recall:    {recall_score(y_test, test_preds, average='weighted', zero_division=0):.5f}")

=== ЛУЧШИЕ ПАРАМЕТРЫ СТЕКИНГА (по Validation) ===
n_splits:    4
passthrough: True
Val Accuracy:  0.9111
Val Precision: 0.9133

=== ФИНАЛЬНЫЕ МЕТРИКИ НА TEST SET ===
Test Accuracy:  0.9053
Test Precision: 0.90844
Test Recall:    0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [61]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

In [62]:
svc_model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
tree_model = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
forest_model = RandomForestClassifier(class_weight='balanced', criterion='entropy', max_depth=24, n_estimators=100, random_state=21)

In [63]:
base_ensemble = VotingClassifier(
    estimators=[
            ('svc', svc_model),
            ('dt', tree_model),
            ('rf', forest_model)
        ],
    voting='soft',
    weights=[4, 1,4]
)
base_ensemble.fit(X_train, y_train)

preds = base_ensemble.predict(X_test)

print("--- ПУНКТ 1: Базовые метрики на TECTE ---")
print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, preds, average='weighted'):.4f}\n")


--- ПУНКТ 1: Базовые метрики на TECTE ---
Accuracy:  0.9053
Precision: 0.9088
Recall:    0.9053



In [64]:
df_results = X_test.copy()
df_results['weekday_true'] = y_test
df_results['weekday_pred'] = preds
df_results['error'] = (df_results['weekday_true'] != df_results['weekday_pred']).astype(int)
user_cols = [col for col in X_test.columns if col.startswith('uid_')]
df_results['user'] = X_test[user_cols].idxmax(axis=1).str.replace('uid_', '')
lab_cols = [col for col in X_test.columns if col.startswith('labname_')]
df_results['labname'] = X_test[lab_cols].idxmax(axis=1).str.replace('labname_', '')
def error_analysis(df, column):
    grouped = df.groupby(column).agg(
        total=('error', 'count'),
        errors=('error', 'sum')
    )
    grouped['error_rate'] = grouped['errors'] / grouped['total']
    return grouped.sort_values(by='error_rate', ascending=False)
weekday_errors = error_analysis(df_results, 'weekday_true')
user_errors = error_analysis(df_results, 'user')
labname_errors = error_analysis(df_results, 'labname')
print(weekday_errors)
print(user_errors)
print(labname_errors)

              total  errors  error_rate
weekday_true                           
0                27       8    0.296296
5                54       7    0.129630
4                21       2    0.095238
6                71       6    0.084507
1                55       4    0.072727
2                30       2    0.066667
3                80       3    0.037500
         total  errors  error_rate
user                              
user_6       4       2    0.500000
user_17      7       2    0.285714
user_3      14       3    0.214286
user_16      5       1    0.200000
user_18      6       1    0.166667
user_27      6       1    0.166667
user_19     19       3    0.157895
user_2      28       4    0.142857
user_30      8       1    0.125000
user_13     17       2    0.117647
user_4      27       3    0.111111
user_31     18       2    0.111111
user_14     31       3    0.096774
user_25     22       2    0.090909
user_29     11       1    0.090909
user_24     11       1    0.090909
user_1    

In [66]:
joblib.dump(voting_model, "best_model_of_ex03(voting_model).pkl")

['best_model_of_ex03(voting_model).pkl']